# Bakery data profile

This notebook inventories the available source files before any cleaning or modelling. It measures missing values and absent rows separately and checks whether joining by date or month would keep every sales row. The two inflation lags are coverage checks for options the team is considering.

In [1]:
import csv
import hashlib
import io
from collections import Counter
from datetime import date, timedelta
from pathlib import Path
from statistics import mean, median
from urllib.request import urlopen

root = Path.cwd()
if root.name == "1_DatasetCharacteristics":
    root = root.parent
base = "https://raw.githubusercontent.com/opencampus-sh/intro-to-data-science-and-ml/main/data/bakery/"
files = {
    "sales": "umsatzdaten_gekuerzt.csv",
    "weather": "wetter.csv",
    "kiel": "kiwo.csv",
    "inflation": "inflation_germany_monthly.csv",
}

def load(source, filename):
    if source == "inflation":
        content = (root / "0_DataPreparation" / filename).read_bytes()
    else:
        with urlopen(base + filename, timeout=30) as response:
            content = response.read()
    print(source, "SHA-256", hashlib.sha256(content).hexdigest())
    return list(csv.DictReader(io.StringIO(content.decode("utf-8-sig"))))

data = {source: load(source, filename) for source, filename in files.items()}
dates = {
    source: {date.fromisoformat(row["Datum"]) for row in data[source]}
    for source in ("sales", "weather", "kiel")
}
months = {row["month"] for row in data["inflation"]}

def days(start, end):
    return {start + timedelta(days=offset) for offset in range((end - start).days + 1)}

def period(value, lag=0):
    year, month = map(int, value[:7].split("-"))
    year, month = divmod(year * 12 + month - 1 - lag, 12)
    return f"{year:04d}-{month + 1:02d}"


sales SHA-256 099a2e609d3ef7286a465f34cb0e62ffb572ca4b454b22aa887c42ad3952dd62
weather SHA-256 14cec82b84eb7abcaa0ebf377e81d122622385e3b71b19e5538c10203af611f8
kiel SHA-256 80a2e3e587061c4a632fd2ee849ec65d37c6a6fb9721f2d6a7db4b7c9587fef8
inflation SHA-256 fb9b27a421b62d6ad0decd3263017e667636d5b7682e18a1c1e998518883acf7


## Source coverage and missingness

Calendar gaps count days with no row in the file. For Kiel Week, the file records event days only.

In [2]:
for source, rows in data.items():
    print("\n", source, len(rows), len(rows[0]))
    for column in rows[0]:
        values = [row[column] for row in rows if row[column] != ""]
        if column not in ("Datum", "month"):
            values = [float(value) for value in values]
        print(column, "range", min(values), max(values), "missing", len(rows) - len(values),
              "distinct", len(set(values)))
    key = "month" if source == "inflation" else "Datum"
    print("duplicate date/month rows", len(rows) - len({row[key] for row in rows}))

sales, weather, kiel, inflation = (data[key] for key in files)
sales_keys = {(row["Datum"], row["Warengruppe"]) for row in sales}
print("duplicate sales keys", len(sales) - len(sales_keys))
print("sales id errors", sum(
    row["id"] != row["Datum"][2:].replace("-", "") + row["Warengruppe"] for row in sales
))
calendar_days = days(min(dates["sales"]), max(dates["sales"]))
for source, observed in dates.items():
    span = days(min(observed), max(observed))
    gaps = span - observed
    print(source, "distinct days", len(observed), "span days", len(span), "gaps", len(gaps),
          "gaps by year", sorted(Counter(day.year for day in gaps).items()))
print("no sales dates", ", ".join(map(str, sorted(calendar_days - dates["sales"]))))
weather_gaps = days(min(dates["weather"]), max(dates["weather"])) - dates["weather"]
print("weather gaps in sales span", sorted(map(str, weather_gaps & calendar_days)))
print("Kiel Week dates by year", sorted(Counter(day.year for day in dates["kiel"]).items()))
print("missing inflation months", sorted({
    f"{year}-{month:02d}" for year in range(2012, 2020) for month in range(1, 13)
    if f"{year}-{month:02d}" <= "2019-07"
} - months))
for column in ("inflation_all_items_pct", "inflation_bread_cereals_pct"):
    print(column, "blank months", [row["month"] for row in inflation if row[column] == ""])
for group in map(str, range(1, 7)):
    rows = [row for row in sales if row["Warengruppe"] == group]
    group_days = {date.fromisoformat(row["Datum"]) for row in rows}
    absent = dates["sales"] - group_days
    values = [float(row["Umsatz"]) for row in rows]
    print("group", group, len(rows), "absent on sales dates", len(absent),
          "months present", sorted({day.month for day in group_days}),
          "first/last", min(group_days), max(group_days),
          "turnover min/median/max", min(values), round(median(values), 2), max(values),
          "nonpositive", sum(value <= 0 for value in values),
          "no seven-day lag", sum(
              ((day - timedelta(days=7)).isoformat(), group) not in sales_keys for day in group_days
          ))
    if group != "6":
        print("absence years", sorted(Counter(day.year for day in absent).items()))
for column in ("Bewoelkung", "Wettercode"):
    for missing in (True, False):
        rows = [row for row in weather if (row[column] == "") == missing]
        print(column, "blank" if missing else "present", len(rows),
              "by month", sorted(Counter(int(row["Datum"][5:7]) for row in rows).items()),
              "mean temp/wind", *(round(mean(float(row[name]) for row in rows), 2)
                                  for name in ("Temperatur", "Windgeschwindigkeit")))
print("weather code values", sorted({int(row["Wettercode"]) for row in weather if row["Wettercode"]}))


 sales 9334 4
id range 1307011.0 1807315.0 missing 0 distinct 9334
Datum range 2013-07-01 2018-07-31 missing 0 distinct 1819
Warengruppe range 1.0 6.0 missing 0 distinct 6
Umsatz range 7.0512010098527 1879.46183076734 missing 0 distinct 9334
duplicate date/month rows 7515

 weather 2601 5
Datum range 2012-01-01 2019-08-01 missing 0 distinct 2601
Bewoelkung range 0.0 8.0 missing 10 distinct 9
Temperatur range -10.25 32.671428 missing 0 distinct 1230
Windgeschwindigkeit range 3.0 35.0 missing 0 distinct 28
Wettercode range 0.0 95.0 missing 669 distinct 41
duplicate date/month rows 0

 kiel 72 2
Datum range 2012-06-16 2019-06-30 missing 0 distinct 72
KielerWoche range 1.0 1.0 missing 0 distinct 1
duplicate date/month rows 0

 inflation 91 5
month range 2012-01 2019-07 missing 0 distinct 91
hicp_all_items range 95.7 106.4 missing 0 distinct 57
hicp_bread_cereals range 93.2 105.3 missing 0 distinct 63
inflation_all_items_pct range -0.51 2.63 missing 12 distinct 59
inflation_bread_cereals_p

## Join coverage and evaluation periods

A left join starts from sales and keeps its rows even when a matching source record is absent.

In [3]:
weather_by_date = {row["Datum"]: row for row in weather}
for column in ("Bewoelkung", "Wettercode"):
    blank_days = {key for key, row in weather_by_date.items() if not row[column]}
    print(column, "matched blank days and sales rows",
          sum(day.isoformat() in blank_days for day in dates["sales"]),
          sum(row["Datum"] in blank_days for row in sales))
for source, right in (("weather", weather), ("kiel", kiel)):
    keys = Counter(row["Datum"] for row in right)
    left = {row["Datum"] for row in sales}
    shared = left & keys.keys()
    print("join", source, "left keys", len(shared), "/", len(left),
          "right keys", len(shared), "/", len(keys),
          "sales rows matched", sum(row["Datum"] in keys for row in sales),
          "left join rows", sum(max(1, keys[row["Datum"]]) for row in sales))
for lag in (1, 2):
    keys = Counter(row["month"] for row in inflation)
    left = {period(row["Datum"], lag) for row in sales}
    print("inflation lag", lag, "left keys", len(left & keys.keys()), "/", len(left),
          "right keys", len(left & keys.keys()), "/", len(keys),
          "sales rows matched", sum(period(row["Datum"], lag) in keys for row in sales),
          "left join rows", sum(max(1, keys[period(row["Datum"], lag)]) for row in sales))
for name, start, end in (
    ("train", "2013-07-01", "2017-07-31"), ("validation", "2017-08-01", "2018-07-31")
):
    rows = [row for row in sales if start <= row["Datum"] <= end]
    observed = {row["Datum"] for row in rows}
    print(name, len(rows), "dates", len(observed), "expected",
          len(days(date.fromisoformat(start), date.fromisoformat(end))),
          "no baseline", sum(
              ((date.fromisoformat(row["Datum"]) - timedelta(days=7)).isoformat(), row["Warengruppe"])
              not in sales_keys for row in rows
          ))


Bewoelkung matched blank days and sales rows 9 54
Wettercode matched blank days and sales rows 456 2309
join weather left keys 1816 / 1819 right keys 1816 / 2601 sales rows matched 9318 left join rows 9334
join kiel left keys 45 / 1819 right keys 45 / 72 sales rows matched 223 left join rows 9334
inflation lag 1 left keys 61 / 61 right keys 61 / 91 sales rows matched 9334 left join rows 9334
inflation lag 2 left keys 61 / 61 right keys 61 / 91 sales rows matched 9334 left join rows 9334
train 7493 dates 1462 expected 1492 no baseline 248
validation 1841 dates 357 expected 365 no baseline 42
